# UC CosMx WTX: NicheNet ligand-regulator pairs in OGN⁺RSPO3⁺ fibroblasts

Builds the table of NicheNet-predicted regulators (with their upstream SELENOP⁺MMP9⁺SPP1⁺ macrophage ligands) that `09_ligand_regulator_cooccurrence.py` tests in tissue.

1. Load the annotated UC WTX object (`01_1k_processing_imputation_comparison.ipynb`, labels from `03_aucell_annotation.R`) and keep the two inflamed UC patients.
2. Load the NicheNet results of `uc_scrna/14_nichenet_macrophage_to_fibroblast.R` (ligand activities, ligand-regulator predictions, predicted regulators, regulator-target network; saved with the `ogn_spp1_mac_` prefix).
3. Split OGN⁺RSPO3⁺ fibroblasts by distance to the nearest Mac S+M+S+ (near < 250 px) and compare each predicted regulator between near and far fibroblasts, by expression and by the score of its NicheNet targets.
4. Merge both tables into `ogn_spp1_mac_nichenet_tf_spatial_validation_combined_v2.csv` (regulator, upstream ligands, expression and activity statistics).

**Output:** `ogn_spp1_mac_nichenet_tf_spatial_validation_combined_v2.csv` (input of `09`).

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
from scipy import stats
from scipy.spatial import cKDTree

In [2]:
adata_wtx=sc.read_h5ad('/share/fsmresfiles/UC/AtoMx/UC_1k_6k_wholetrans/whole_trans/Processed_merged/v7/h5ad_obj/qc_wtx_v7_merged_anno.h5ad')

/data/ydn4687/geneformer_env/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [139]:
adata_wtx=adata_wtx[adata_wtx.obs['patient'].isin(['I0261','I0331'])]

In [550]:
# Load NicheNet results
ligand_tf_predictions = pd.read_csv("/share/fsmresfiles/UC/scRNA-seq/merged_9p_wip/ogn_spp1_mac_nichenet_ligand_tf_predictions.csv")
predicted_tfs = pd.read_csv("/share/fsmresfiles/UC/scRNA-seq/merged_9p_wip/ogn_spp1_mac_nichenet_predicted_tfs.csv")
tf_target_network = pd.read_csv("/share/fsmresfiles/UC/scRNA-seq/merged_9p_wip/ogn_spp1_mac_nichenet_tf_target_network.csv")
ligand_activities = pd.read_csv("/share/fsmresfiles/UC/scRNA-seq/merged_9p_wip/ogn_spp1_mac_nichenet_ligand_activities.csv")

In [556]:
# Cell masks for proximity
ogn_fib_mask = adata_wtx.obs['aucell_cell_type_short'] == 'OGN+RSPO3+ Fib'
spp1_mac_mask = adata_wtx.obs['aucell_cell_type_short'] == 'Mac S+M+S+'


In [557]:
all_ogn_coords = adata_wtx[ogn_fib_mask].obsm['spatial_fov']
all_spp1_coords = adata_wtx[spp1_mac_mask].obsm['spatial_fov']
# Get coordinates

In [558]:
tree = cKDTree(all_spp1_coords)
distances, _ = tree.query(all_ogn_coords, k=1)

In [560]:
# Create OGN+ fibroblast subset with distance info
adata_ogn = adata_wtx[ogn_fib_mask].copy()
adata_ogn.obs['distance_to_spp1_mac'] = distances

In [561]:
threshold = 250  # pixels (spatial_fov coordinates)
adata_ogn.obs['proximity_to_spp1_mac'] = 'far'
adata_ogn.obs.loc[adata_ogn.obs['distance_to_spp1_mac'] < threshold, 'proximity_to_spp1_mac'] = 'near'

In [563]:
# Test TF expression near vs far
predicted_tf_list = predicted_tfs['tf'].tolist()

In [564]:
tf_expression_results = []

for tf in predicted_tf_list:
    if tf in adata_ogn.var_names:
        near_expr = adata_ogn[adata_ogn.obs['proximity_to_spp1_mac'] == 'near', tf].X
        far_expr = adata_ogn[adata_ogn.obs['proximity_to_spp1_mac'] == 'far', tf].X
        
        if hasattr(near_expr, 'toarray'):
            near_expr = near_expr.toarray().flatten()
            far_expr = far_expr.toarray().flatten()
        
        stat_val, pval = stats.ttest_ind(near_expr, far_expr)
        
        # Get upstream ligands from NicheNet
        upstream = ligand_tf_predictions[ligand_tf_predictions['tf'] == tf]['ligand'].unique()
        
        tf_expression_results.append({
            'TF': tf,
            'mean_near': near_expr.mean(),
            'mean_far': far_expr.mean(),
            'log2FC': np.log2((near_expr.mean() + 1) / (far_expr.mean() + 1)),
            'pvalue': pval,
            'pct_near': (near_expr > 0).sum() / len(near_expr) * 100,
            'pct_far': (far_expr > 0).sum() / len(far_expr) * 100,
            'upstream_ligands': ','.join(upstream[:3]) if len(upstream) > 0 else 'none'
        })


In [565]:
tf_expr_df = pd.DataFrame(tf_expression_results)
tf_expr_df['padj'] = stats.false_discovery_control(tf_expr_df['pvalue'])
tf_expr_df = tf_expr_df.sort_values('log2FC', ascending=False)

In [567]:
# Test TF ACTIVITY using target genes
tf_activity_results = []

for tf in predicted_tf_list:
    # Get target genes for this TF from NicheNet
    targets = tf_target_network[tf_target_network['from'] == tf]['to'].tolist()
    targets_present = [t for t in targets if t in adata_ogn.var_names]
    
    if len(targets_present) < 3:
        continue
    
    # Score TF activity
    sc.tl.score_genes(
        adata_ogn,
        gene_list=targets_present,
        score_name=f'{tf}_activity',
        use_raw=False
    )
    
    near_activity = adata_ogn[adata_ogn.obs['proximity_to_spp1_mac'] == 'near'].obs[f'{tf}_activity']
    far_activity = adata_ogn[adata_ogn.obs['proximity_to_spp1_mac'] == 'far'].obs[f'{tf}_activity']
    
    stat_val, pval = stats.ttest_ind(near_activity, far_activity)
    
    upstream = ligand_tf_predictions[ligand_tf_predictions['tf'] == tf]['ligand'].unique()
    
    tf_activity_results.append({
        'TF': tf,
        'n_targets': len(targets_present),
        'mean_activity_near': near_activity.mean(),
        'mean_activity_far': far_activity.mean(),
        'activity_diff': near_activity.mean() - far_activity.mean(),
        'pvalue': pval,
        'upstream_ligands': ','.join(upstream[:3]) if len(upstream) > 0 else 'none'
    })


In [ ]:
tf_activity_df = pd.DataFrame(tf_activity_results)
tf_activity_df['padj'] = stats.false_discovery_control(tf_activity_df['pvalue'])
tf_activity_df = tf_activity_df.sort_values('activity_diff', ascending=False)


In [ ]:
# Combine results
combined = tf_expr_df[['TF', 'log2FC', 'padj', 'pvalue','upstream_ligands']].merge(
    tf_activity_df[['TF', 'activity_diff', 'padj','pvalue']],
    on='TF',
    suffixes=('_expr', '_activity')
)

In [ ]:
combined.to_csv('/share/fsmresfiles/UC/scRNA-seq/merged_9p_wip/ogn_spp1_mac_nichenet_tf_spatial_validation_combined_v2.csv', index=False)